In [1]:
import pandas as pd
import datetime as dt
import numpy as np

In [2]:
ace_mapping = pd.read_csv(
    "/mnt/disks/data/ACE/ace_mapping.csv", parse_dates=[0, 1], header=0
)
dscovr_mapping = pd.read_csv(
    "/mnt/disks/data/DSCOVR/dscovr_mapping.csv", parse_dates=[0, 1], header=0
)

In [3]:
def filter_target_times(mapping, start_time, end_time, horizon=90):
    """
    This function returns all Target_times within the mapping data frame that are within the time horizon of the start and end times.

    Inputs:
    mapping: pandas dataframe
    start_time: datetime object
    end_time: datetime object
    horizon: int (default = 90)

    Outputs:
    mapping_filtered: pandas dataframe
    """

    mapping_filtered = mapping[
        (
            (mapping["Target_time"] >= start_time - pd.Timedelta(minutes=horizon))
            & (mapping["Target_time"] <= end_time + pd.Timedelta(minutes=horizon))
        )
    ]
    mapping_remaining = mapping[
        ~(
            (mapping["Target_time"] >= start_time - pd.Timedelta(minutes=horizon))
            & (mapping["Target_time"] <= end_time + pd.Timedelta(minutes=horizon))
        )
    ]

    return mapping_filtered, mapping_remaining

In [4]:
benchmark_events = [
    [dt.datetime(2003, 10, 29, 0, 0), dt.datetime(2003, 10, 30, 23, 59)],
    [dt.datetime(2006, 12, 29, 0, 0), dt.datetime(2006, 12, 30, 23, 59)],
    [dt.datetime(2001, 8, 31, 0, 0), dt.datetime(2001, 9, 1, 23, 59)],
    [dt.datetime(2005, 8, 31, 0, 0), dt.datetime(2005, 9, 1, 23, 59)],
    [dt.datetime(2010, 4, 5, 0, 0), dt.datetime(2010, 4, 6, 23, 59)],
    [dt.datetime(2011, 8, 5, 0, 0), dt.datetime(2011, 8, 6, 23, 59)],
    [dt.datetime(2015, 3, 17, 0, 0), dt.datetime(2015, 3, 19, 23, 59)],
    [dt.datetime(2017, 9, 27, 0, 0), dt.datetime(2017, 9, 28, 23, 59)],
    [dt.datetime(2024, 5, 9, 0, 0), dt.datetime(2024, 5, 12, 23, 59)],
]

In [5]:
ace_mapping_train = ace_mapping
ace_mapping_test = pd.DataFrame()
dscovr_mapping_train = dscovr_mapping
dscovr_mapping_test = pd.DataFrame()
for i in benchmark_events:
    ace_mapping_filtered, ace_mapping_train = filter_target_times(
        ace_mapping_train, i[0], i[1]
    )
    ace_mapping_test = pd.concat([ace_mapping_test, ace_mapping_filtered])
    dscovr_mapping_filtered, dscovr_mapping_train = filter_target_times(
        dscovr_mapping_train, i[0], i[1]
    )
    dscovr_mapping_test = pd.concat([dscovr_mapping_test, dscovr_mapping_filtered])

In [6]:
dscovr_mapping_test.to_csv(
    "/mnt/disks/data/DSCOVR/dscovr_mapping_test.csv", index=False
)
dscovr_mapping_train.to_csv(
    "/mnt/disks/data/DSCOVR/dscovr_mapping_train_full.csv", index=False
)
ace_mapping_test.to_csv("/mnt/disks/data/ACE/ace_mapping_test.csv", index=False)
ace_mapping_train.to_csv("/mnt/disks/data/ACE/ace_mapping_train_full.csv", index=False)

In [16]:
def split_train_val(mapping, horizon=90, split=0.2, seed=42):
    """
    This function returns a training and validation set from the mapping data frame.

    Inputs:
    mapping: pandas dataframe
    horizon: int (default = 90)
    split: float (default = 0.2)
    seed: int (default = 42)

    Outputs:
    mapping_train: pandas dataframe
    mapping_val: pandas dataframe
    """

    n_samples = np.arange(len(mapping) / (24 * 60))
    np.random.seed(seed)
    n_rand = (
        np.random.choice(n_samples, int(len(n_samples) * split), replace=False).astype(
            int
        )
        * 24
        * 60
    )
    val_selection = mapping.iloc[list(n_rand)]

    # ensure validation set does not overlap with horizon of training set
    mapping_train = mapping
    mapping_val = pd.DataFrame()
    mapping_val_full = pd.DataFrame()
    for i in val_selection["Target_time"]:
        mapping_filtered, mapping_train = filter_target_times(
            mapping_train, i, i + pd.Timedelta(minutes=int(24 * 60)), horizon
        )
        mapping_filtered_val = mapping_filtered[
            (mapping_filtered["Target_time"] >= i)
            & (
                mapping_filtered["Target_time"]
                <= i + pd.Timedelta(minutes=int(24 * 60))
            )
        ]
        mapping_val = pd.concat([mapping_val, mapping_filtered_val])
        mapping_val_full = pd.concat([mapping_val_full, mapping_filtered])

    return mapping_val, mapping_train, mapping_val_full

In [27]:
ace_val, ace_train, ace_val_full = split_train_val(ace_mapping_train, split=0.2)

In [28]:
dscover_val, dscover_train, dscover_val_full = split_train_val(
    dscovr_mapping_train, split=0.2
)

In [29]:
(len(dscover_val) + len(ace_val)) / (
    len(dscover_train) + len(dscover_val) + len(ace_train) + len(ace_val)
)

0.1326131379585327

In [30]:
ace_val.to_csv("/mnt/disks/data/ACE/ace_mapping_val.csv", index=False)
ace_train.to_csv("/mnt/disks/data/ACE/ace_mapping_train.csv", index=False)
dscover_val.to_csv("/mnt/disks/data/DSCOVR/dscovr_mapping_val.csv", index=False)
dscover_train.to_csv("/mnt/disks/data/DSCOVR/dscovr_mapping_train.csv", index=False)